In [1]:
%pip install mysql-connector-python pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import mysql.connector
import pandas as pd
from IPython.display import display

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    autocommit=True
)
setup_cursor = connection.cursor()
setup_cursor.execute("CREATE DATABASE IF NOT EXISTS week2_sql_practicals")
setup_cursor.execute("USE week2_sql_practicals")
setup_cursor.execute("SELECT VERSION()")
mysql_version = setup_cursor.fetchone()[0]
if int(mysql_version.split(".")[0]) < 8:
    raise RuntimeError("MySQL 8.0 or newer is required")
setup_cursor.close()

def run(sql, params=None):
    cur = connection.cursor(dictionary=True, buffered=True)
    if params is None:
        cur.execute(sql)
    else:
        cur.execute(sql, params)
    if cur.with_rows:
        rows = cur.fetchall()
        display(pd.DataFrame(rows))
    cur.close()

def call(sql, params=None):
    cur = connection.cursor(dictionary=True, buffered=True)
    if params is None:
        cur.execute(sql)
    else:
        cur.execute(sql, params)
    while True:
        if cur.with_rows:
            rows = cur.fetchall()
            display(pd.DataFrame(rows))
        if not cur.nextset():
            break
    cur.close()

In [3]:
run("DROP TABLE IF EXISTS book_review")
run("DROP TABLE IF EXISTS adaptation")
run("DROP TABLE IF EXISTS book")
run("DROP TABLE IF EXISTS author")

run("CREATE TABLE author (author_id INT PRIMARY KEY, author_name VARCHAR(100) NOT NULL)")
run("CREATE TABLE book (book_id INT PRIMARY KEY, author_id INT NOT NULL, title VARCHAR(150) NOT NULL, publish_year INT NOT NULL, FOREIGN KEY (author_id) REFERENCES author(author_id))")
run("CREATE TABLE adaptation (book_id INT PRIMARY KEY, adaptation_type VARCHAR(20) NOT NULL, title VARCHAR(150) NOT NULL, release_year INT NOT NULL, rating DECIMAL(3,2) NOT NULL, FOREIGN KEY (book_id) REFERENCES book(book_id))")
run("CREATE TABLE book_review (review_id INT PRIMARY KEY, book_id INT NOT NULL, rating DECIMAL(3,2) NOT NULL, FOREIGN KEY (book_id) REFERENCES book(book_id))")

run("INSERT INTO author VALUES (%s, %s), (%s, %s), (%s, %s), (%s, %s)", (1, "Anita Rao", 2, "Dev Malhotra", 3, "Kiran Sethi", 4, "Leela Thomas"))
run("INSERT INTO book VALUES (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s)", (101,1,"The Monsoon Letters",2015, 102,1,"City of Lanterns",2018, 103,2,"Code of the Hills",2012, 104,3,"The Last Tea Garden",2019, 105,4,"Silent Harbour",2016, 106,2,"Paper Kites",2020))
run("INSERT INTO adaptation VALUES (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s)", (101,"Movie","Monsoon Letters",2017,8.10, 102,"Series","Lanterns",2024,7.90, 103,"Movie","Hill Code",2014,7.00, 104,"Series","Tea Garden",2022,8.60, 105,"Movie","Harbour",2018,6.50, 106,"Movie","Kites",2021,7.40))
run("INSERT INTO book_review VALUES (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s)", (1,101,7, 2,101,8, 3,102,6, 4,102,7, 5,103,8, 6,103,7, 7,103,9, 8,104,8, 9,104,8, 10,105,6, 11,105,7, 12,106,7, 13,106,7, 14,106,7))

run("""
SELECT b.title AS book_title,
       a.title AS adaptation_title,
       a.release_year AS adaptation_year,
       b.publish_year AS publication_year,
       au.author_name,
       ROUND(AVG(br.rating), 2) AS book_rating,
       a.rating AS adaptation_rating
FROM book AS b
JOIN author AS au ON au.author_id = b.author_id
JOIN adaptation AS a ON a.book_id = b.book_id
JOIN book_review AS br ON br.book_id = b.book_id
WHERE a.rating > (SELECT AVG(br2.rating) FROM book_review AS br2 WHERE br2.book_id = b.book_id)
  AND a.release_year BETWEEN b.publish_year AND b.publish_year + 4
GROUP BY b.book_id, b.title, a.title, a.release_year, b.publish_year, au.author_name, a.rating
ORDER BY b.title
""")

,book_title,adaptation_title,adaptation_year,publication_year,author_name,book_rating,adaptation_rating
0,Paper Kites,Kites,2021,2020,Dev Malhotra,7.00,7.40
1,The Last Tea Garden,Tea Garden,2022,2019,Kiran Sethi,8.00,8.60
2,The Monsoon Letters,Monsoon Letters,2017,2015,Anita Rao,7.50,8.10


In [4]:
run("DROP PROCEDURE IF EXISTS sp_customer_orders")
run("DROP FUNCTION IF EXISTS fn_format_us_date")
run("DROP TABLE IF EXISTS order_items")
run("DROP TABLE IF EXISTS orders")
run("DROP TABLE IF EXISTS customers")

run("CREATE TABLE customers (customer_id VARCHAR(5) PRIMARY KEY, customer_name VARCHAR(100) NOT NULL, city VARCHAR(50) NOT NULL, signup_date DATE NOT NULL)")
run("CREATE TABLE orders (order_id INT PRIMARY KEY, customer_id VARCHAR(5) NOT NULL, order_date DATE NOT NULL, status VARCHAR(20) NOT NULL, FOREIGN KEY (customer_id) REFERENCES customers(customer_id))")
run("CREATE TABLE order_items (order_id INT NOT NULL, product_id VARCHAR(5) NOT NULL, qty INT NOT NULL, unit_price DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, product_id), FOREIGN KEY (order_id) REFERENCES orders(order_id))")

run("INSERT INTO customers VALUES (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s)", ("C01","Rahul Mehta","Jaipur","2025-11-02", "C02","Priya Nair","Kochi","2025-12-15", "C03","Arjun Das","Kolkata","2026-01-08", "C04","Fatima Sheikh","Mumbai","2026-01-20", "C05","Karan Malhotra","Delhi","2026-02-11", "C06","Sneha Kulkarni","Pune","2026-02-25"))
run("INSERT INTO orders VALUES (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s)", (1001,"C01","2026-03-02","Delivered", 1002,"C02","2026-03-03","Delivered", 1003,"C01","2026-03-10","Delivered", 1004,"C03","2026-03-12","Cancelled", 1005,"C04","2026-03-15","Shipped", 1006,"C05","2026-03-18","Delivered", 1007,"C01","2026-04-01","Shipped", 1008,"C06","2026-04-03","Pending"))
run("INSERT INTO order_items VALUES (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s)", (1001,"P1",2,799, 1001,"P3",5,120, 1002,"P2",1,1999, 1003,"P4",2,250, 1003,"P6",1,499, 1004,"P5",1,1499, 1005,"P1",1,799, 1005,"P2",1,1999, 1005,"P3",10,120, 1006,"P6",3,499, 1007,"P5",1,1499, 1008,"P4",4,250))

run("""
CREATE FUNCTION fn_format_us_date(p_date DATETIME)
RETURNS VARCHAR(10)
DETERMINISTIC
NO SQL
RETURN DATE_FORMAT(p_date, '%m/%d/%Y')
""")

run("""
CREATE PROCEDURE sp_customer_orders(
    IN p_customer_id VARCHAR(5),
    OUT p_order_count INT,
    OUT p_total_value DECIMAL(12,2)
)
BEGIN
    SET p_order_count = 0;
    SET p_total_value = 0;

    IF EXISTS (SELECT 1 FROM customers WHERE customer_id = p_customer_id) THEN
        SELECT o.order_id,
               fn_format_us_date(o.order_date) AS order_date_us,
               o.status,
               COALESCE(SUM(oi.qty * oi.unit_price), 0) AS order_value
        FROM orders AS o
        LEFT JOIN order_items AS oi ON oi.order_id = o.order_id
        WHERE o.customer_id = p_customer_id
          AND o.status <> 'Cancelled'
        GROUP BY o.order_id, o.order_date, o.status
        ORDER BY o.order_id;

        SELECT COUNT(*), COALESCE(SUM(order_value), 0)
        INTO p_order_count, p_total_value
        FROM (
            SELECT o.order_id, COALESCE(SUM(oi.qty * oi.unit_price), 0) AS order_value
            FROM orders AS o
            LEFT JOIN order_items AS oi ON oi.order_id = o.order_id
            WHERE o.customer_id = p_customer_id
              AND o.status <> 'Cancelled'
            GROUP BY o.order_id
        ) AS order_totals;
    END IF;
END
""")

call("CALL sp_customer_orders('C01', @order_count, @total_value)")
run("SELECT @order_count AS order_count, @total_value AS total_value")
call("CALL sp_customer_orders('C99', @order_count, @total_value)")
run("SELECT @order_count AS order_count, @total_value AS total_value")
run("SELECT fn_format_us_date('2006-11-21 23:34:05') AS formatted_date")

,order_id,order_date_us,status,order_value
0,1001,03/02/2026,Delivered,2198.00
1,1003,03/10/2026,Delivered,999.00
2,1007,04/01/2026,Shipped,1499.00


,order_count,total_value
0,3,4696.00


,order_count,total_value
0,0,0.00


,formatted_date
0,11/21/2006


In [5]:
run("DROP PROCEDURE IF EXISTS sp_dateproc")
run("DROP TABLE IF EXISTS datepro_data")
run("DROP TABLE IF EXISTS datepro")

run("CREATE TABLE datepro (id INT PRIMARY KEY, Dateint INT NOT NULL, Datentime VARCHAR(40) NOT NULL)")
run("CREATE TABLE datepro_data (id INT NOT NULL, Dateint INT NOT NULL, Datentime VARCHAR(16) NOT NULL, Dates DATE NOT NULL, PRIMARY KEY (id, Dates))")
run("INSERT INTO datepro VALUES (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s)", (101,20220510,"2022-05-10T09:30:00+05:30", 102,20220227,"2022-02-27T18:45:00+05:30", 103,20220925,"2022-09-25T12:16:00+05:30", 104,20221229,"2022-12-29T07:05:00+05:30", 105,20221228,"2022-12-28T11:29:00+05:30"))

run("""
CREATE PROCEDURE sp_dateproc(IN p_ids VARCHAR(100))
BEGIN
    DELETE FROM datepro_data;

    INSERT INTO datepro_data (id, Dateint, Datentime, Dates)
    WITH RECURSIVE selected_dates AS (
        SELECT id,
               Dateint,
               LEFT(REPLACE(Datentime, 'T', ' '), 16) AS clean_datentime,
               STR_TO_DATE(CAST(Dateint AS CHAR), '%Y%m%d') AS start_date
        FROM datepro
        WHERE FIND_IN_SET(CAST(id AS CHAR), p_ids) > 0
    ), calendar AS (
        SELECT id,
               Dateint,
               clean_datentime,
               start_date AS date_value,
               LAST_DAY(start_date) AS last_date
        FROM selected_dates
        UNION ALL
        SELECT id,
               Dateint,
               clean_datentime,
               DATE_ADD(date_value, INTERVAL 1 DAY),
               last_date
        FROM calendar
        WHERE date_value < last_date
    )
    SELECT id, Dateint, clean_datentime, date_value
    FROM calendar;

    SELECT id, Dateint, Datentime, Dates
    FROM datepro_data
    ORDER BY id, Dates;
END
""")

call("CALL sp_dateproc('101,102,104')")
run("SELECT COUNT(*) AS total_rows FROM datepro_data")

,id,Dateint,Datentime,Dates
0,101,20220510,2022-05-10 09:30,2022-05-10
1,101,20220510,2022-05-10 09:30,2022-05-11
2,101,20220510,2022-05-10 09:30,2022-05-12
3,101,20220510,2022-05-10 09:30,2022-05-13
4,101,20220510,2022-05-10 09:30,2022-05-14
5,101,20220510,2022-05-10 09:30,2022-05-15
6,101,20220510,2022-05-10 09:30,2022-05-16
7,101,20220510,2022-05-10 09:30,2022-05-17
8,101,20220510,2022-05-10 09:30,2022-05-18
9,101,20220510,2022-05-10 09:30,2022-05-19


,total_rows
0,27


In [6]:
run("DROP PROCEDURE IF EXISTS sp_allot_subjects")
run("DROP TABLE IF EXISTS UnallotedStudents")
run("DROP TABLE IF EXISTS Allotments")
run("DROP TABLE IF EXISTS StudentPreference")
run("DROP TABLE IF EXISTS SubjectDetails")
run("DROP TABLE IF EXISTS StudentDetails")

run("CREATE TABLE StudentDetails (StudentId BIGINT PRIMARY KEY, StudentName VARCHAR(100) NOT NULL, GPA DECIMAL(3,2) NOT NULL, Branch VARCHAR(10) NOT NULL, Section VARCHAR(5) NOT NULL)")
run("CREATE TABLE SubjectDetails (SubjectId VARCHAR(10) PRIMARY KEY, SubjectName VARCHAR(100) NOT NULL, MaxSeats INT NOT NULL, RemainingSeats INT NOT NULL)")
run("CREATE TABLE StudentPreference (StudentId BIGINT NOT NULL, SubjectId VARCHAR(10) NOT NULL, Preference INT NOT NULL, PRIMARY KEY (StudentId, SubjectId), UNIQUE (StudentId, Preference), FOREIGN KEY (StudentId) REFERENCES StudentDetails(StudentId), FOREIGN KEY (SubjectId) REFERENCES SubjectDetails(SubjectId))")
run("CREATE TABLE Allotments (SubjectId VARCHAR(10) NOT NULL, StudentId BIGINT NOT NULL, PRIMARY KEY (SubjectId, StudentId), UNIQUE (StudentId), FOREIGN KEY (SubjectId) REFERENCES SubjectDetails(SubjectId), FOREIGN KEY (StudentId) REFERENCES StudentDetails(StudentId))")
run("CREATE TABLE UnallotedStudents (StudentId BIGINT PRIMARY KEY, FOREIGN KEY (StudentId) REFERENCES StudentDetails(StudentId))")

run("INSERT INTO StudentDetails VALUES (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s)", (159103036,"Mohit Agarwal",8.90,"CCE","A", 159103037,"Rohit Agarwal",5.20,"CCE","A", 159103038,"Shohit Garg",7.10,"CCE","B", 159103039,"Mrinal Malhotra",7.90,"CCE","A", 159103040,"Mehreet Singh",5.60,"CCE","A", 159103041,"Arjun Tehlan",9.20,"CCE","B"))
run("INSERT INTO SubjectDetails VALUES (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s)", ("PO1491","Basics of Political Science",60,2, "PO1492","Basics of Accounting",120,1, "PO1493","Basics of Financial Markets",90,1, "PO1494","Eco philosophy",60,1, "PO1495","Automotive Trends",60,0))
run("INSERT INTO StudentPreference VALUES (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s)", (159103036,"PO1491",1, 159103036,"PO1492",2, 159103036,"PO1493",3, 159103036,"PO1494",4, 159103036,"PO1495",5, 159103037,"PO1494",1, 159103037,"PO1495",2, 159103037,"PO1492",3, 159103037,"PO1491",4, 159103037,"PO1493",5, 159103038,"PO1492",1, 159103038,"PO1491",2, 159103038,"PO1494",3, 159103038,"PO1495",4, 159103038,"PO1493",5, 159103039,"PO1491",1, 159103039,"PO1492",2, 159103039,"PO1495",3, 159103039,"PO1493",4, 159103039,"PO1494",5, 159103040,"PO1492",1, 159103040,"PO1494",2, 159103040,"PO1491",3, 159103040,"PO1493",4, 159103040,"PO1495",5, 159103041,"PO1491",1, 159103041,"PO1493",2, 159103041,"PO1492",3, 159103041,"PO1494",4, 159103041,"PO1495",5))

run("""
CREATE PROCEDURE sp_allot_subjects()
BEGIN
    DECLARE v_done INT DEFAULT 0;
    DECLARE v_student_id BIGINT;
    DECLARE v_subject_id VARCHAR(10);
    DECLARE v_current_student BIGINT DEFAULT NULL;
    DECLARE v_allocated TINYINT DEFAULT 0;

    DECLARE preference_cursor CURSOR FOR
        SELECT sd.StudentId, sp.SubjectId
        FROM StudentDetails AS sd
        JOIN StudentPreference AS sp ON sp.StudentId = sd.StudentId
        ORDER BY sd.GPA DESC, sd.StudentId, sp.Preference;

    DECLARE CONTINUE HANDLER FOR NOT FOUND SET v_done = 1;

    DELETE FROM Allotments;
    DELETE FROM UnallotedStudents;
    UPDATE SubjectDetails SET RemainingSeats = MaxSeats;

    UPDATE SubjectDetails SET RemainingSeats = CASE SubjectId
        WHEN 'PO1491' THEN 2
        WHEN 'PO1492' THEN 1
        WHEN 'PO1493' THEN 1
        WHEN 'PO1494' THEN 1
        WHEN 'PO1495' THEN 0
        ELSE RemainingSeats
    END;

    OPEN preference_cursor;

    preference_loop: LOOP
        FETCH preference_cursor INTO v_student_id, v_subject_id;
        IF v_done = 1 THEN
            LEAVE preference_loop;
        END IF;

        IF v_current_student IS NULL OR v_current_student <> v_student_id THEN
            IF v_current_student IS NOT NULL AND v_allocated = 0 THEN
                INSERT INTO UnallotedStudents (StudentId) VALUES (v_current_student);
            END IF;
            SET v_current_student = v_student_id;
            SET v_allocated = 0;
        END IF;

        IF v_allocated = 0 THEN
            UPDATE SubjectDetails
            SET RemainingSeats = RemainingSeats - 1
            WHERE SubjectId = v_subject_id AND RemainingSeats > 0;

            IF ROW_COUNT() = 1 THEN
                INSERT INTO Allotments (SubjectId, StudentId) VALUES (v_subject_id, v_student_id);
                SET v_allocated = 1;
            END IF;
        END IF;
    END LOOP;

    IF v_current_student IS NOT NULL AND v_allocated = 0 THEN
        INSERT INTO UnallotedStudents (StudentId) VALUES (v_current_student);
    END IF;

    CLOSE preference_cursor;
END
""")

call("CALL sp_allot_subjects()")
run("SELECT a.SubjectId, a.StudentId FROM Allotments AS a JOIN StudentDetails AS sd ON sd.StudentId = a.StudentId ORDER BY sd.GPA DESC, sd.StudentId")
run("SELECT StudentId FROM UnallotedStudents ORDER BY StudentId")
run("SELECT SubjectId, SubjectName, MaxSeats, RemainingSeats FROM SubjectDetails ORDER BY SubjectId")

,SubjectId,StudentId
0,PO1491,159103041
1,PO1491,159103036
2,PO1492,159103039
3,PO1494,159103038
4,PO1493,159103040


,StudentId
0,159103037


,SubjectId,SubjectName,MaxSeats,RemainingSeats
0,PO1491,Basics of Political Science,60,0
1,PO1492,Basics of Accounting,120,0
2,PO1493,Basics of Financial Markets,90,0
3,PO1494,Eco philosophy,60,0
4,PO1495,Automotive Trends,60,0


In [7]:
run("DROP PROCEDURE IF EXISTS sp_record_payment")
run("DROP TABLE IF EXISTS error_log")
run("DROP TABLE IF EXISTS payments")
run("DROP TABLE IF EXISTS order_items")
run("DROP TABLE IF EXISTS orders")

run("CREATE TABLE orders (order_id INT PRIMARY KEY, customer_id VARCHAR(5), order_date DATE NOT NULL, status VARCHAR(20) NOT NULL)")
run("CREATE TABLE order_items (order_id INT NOT NULL, product_id VARCHAR(5) NOT NULL, qty INT NOT NULL, unit_price DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, product_id), FOREIGN KEY (order_id) REFERENCES orders(order_id))")
run("CREATE TABLE payments (payment_id VARCHAR(10) PRIMARY KEY, order_id INT NOT NULL, payment_date DATE NOT NULL, amount DECIMAL(10,2) NOT NULL, method VARCHAR(20) NOT NULL)")
run("CREATE TABLE error_log (log_id INT AUTO_INCREMENT PRIMARY KEY, error_time DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP, proc_name VARCHAR(100) NOT NULL, error_number INT NOT NULL, error_message TEXT NOT NULL, input_values VARCHAR(255) NOT NULL)")

run("INSERT INTO orders VALUES (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s)", (1001,"C01","2026-03-02","Delivered", 1002,"C02","2026-03-03","Delivered", 1003,"C01","2026-03-10","Delivered", 1004,"C03","2026-03-12","Cancelled", 1005,"C04","2026-03-15","Shipped", 1006,"C05","2026-03-18","Delivered", 1007,"C01","2026-04-01","Shipped", 1008,"C06","2026-04-03","Pending"))
run("INSERT INTO order_items VALUES (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s)", (1001,"P1",2,799, 1001,"P3",5,120, 1002,"P2",1,1999, 1003,"P4",2,250, 1003,"P6",1,499, 1004,"P5",1,1499, 1005,"P1",1,799, 1005,"P2",1,1999, 1005,"P3",10,120, 1006,"P6",3,499, 1007,"P5",1,1499, 1008,"P4",4,250))
run("INSERT INTO payments (payment_id, order_id, payment_date, amount, method) VALUES (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s)", ("PM01",1001,"2026-03-02",2198,"UPI", "PM02",1002,"2026-03-03",1000,"Card", "PM03",1002,"2026-03-05",999,"Card", "PM04",1003,"2026-03-10",500,"UPI", "PM05",1005,"2026-03-15",3998,"NetBanking", "PM06",1006,"2026-03-18",1497,"UPI", "PM07",1009,"2026-04-02",750,"UPI", "PM08",1004,"2026-03-12",1499,"Card"))

run("""
CREATE PROCEDURE sp_record_payment(
    IN p_payment_id VARCHAR(10),
    IN p_order_id INT,
    IN p_amount DECIMAL(10,2),
    IN p_method VARCHAR(20)
)
BEGIN
    DECLARE v_order_count INT DEFAULT 0;
    DECLARE v_order_status VARCHAR(20);
    DECLARE v_order_value DECIMAL(12,2) DEFAULT 0;
    DECLARE v_paid_amount DECIMAL(12,2) DEFAULT 0;
    DECLARE v_error_number INT DEFAULT 0;
    DECLARE v_error_message TEXT;

    DECLARE EXIT HANDLER FOR SQLEXCEPTION
    BEGIN
        GET DIAGNOSTICS CONDITION 1
            v_error_number = MYSQL_ERRNO,
            v_error_message = MESSAGE_TEXT;
        ROLLBACK;
        INSERT INTO error_log (proc_name, error_number, error_message, input_values)
        VALUES (
            'sp_record_payment',
            v_error_number,
            v_error_message,
            CONCAT(p_payment_id, ',', p_order_id, ',', p_amount, ',', p_method)
        );
    END;

    START TRANSACTION;

    SELECT COUNT(*) INTO v_order_count
    FROM orders
    WHERE order_id = p_order_id;

    IF v_order_count = 0 THEN
        SIGNAL SQLSTATE '45000'
            SET MYSQL_ERRNO = 50001, MESSAGE_TEXT = 'Order not found';
    END IF;

    SELECT status INTO v_order_status
    FROM orders
    WHERE order_id = p_order_id
    FOR UPDATE;

    IF v_order_status = 'Cancelled' THEN
        SIGNAL SQLSTATE '45000'
            SET MYSQL_ERRNO = 50002, MESSAGE_TEXT = 'Order is cancelled';
    END IF;

    IF p_amount <= 0 THEN
        SIGNAL SQLSTATE '45000'
            SET MYSQL_ERRNO = 50003, MESSAGE_TEXT = 'Amount must be greater than zero';
    END IF;

    SELECT COALESCE(SUM(qty * unit_price), 0) INTO v_order_value
    FROM order_items
    WHERE order_id = p_order_id;

    SELECT COALESCE(SUM(amount), 0) INTO v_paid_amount
    FROM payments
    WHERE order_id = p_order_id;

    IF v_paid_amount + p_amount > v_order_value THEN
        SIGNAL SQLSTATE '45000'
            SET MYSQL_ERRNO = 50004, MESSAGE_TEXT = 'Payment exceeds order value';
    END IF;

    INSERT INTO payments (payment_id, order_id, payment_date, amount, method)
    VALUES (p_payment_id, p_order_id, CURDATE(), p_amount, p_method);

    COMMIT;
END
""")

call("CALL sp_record_payment(%s,%s,%s,%s)", ("PM09",1007,1499,"Card"))
call("CALL sp_record_payment(%s,%s,%s,%s)", ("PM10",1010,500,"UPI"))
call("CALL sp_record_payment(%s,%s,%s,%s)", ("PM11",1004,100,"UPI"))
call("CALL sp_record_payment(%s,%s,%s,%s)", ("PM12",1003,0,"UPI"))
call("CALL sp_record_payment(%s,%s,%s,%s)", ("PM13",1003,600,"UPI"))
call("CALL sp_record_payment(%s,%s,%s,%s)", ("PM14",1003,499,"UPI"))
run("SELECT payment_id, order_id, amount, method FROM payments WHERE payment_id IN ('PM09','PM14') ORDER BY payment_id")
run("SELECT log_id, proc_name, error_number, error_message, input_values FROM error_log ORDER BY log_id")

,payment_id,order_id,amount,method
0,PM09,1007,1499.00,Card
1,PM14,1003,499.00,UPI


,log_id,proc_name,error_number,error_message,input_values
0,1,sp_record_payment,50001,Order not found,"PM10,1010,500.00,UPI"
1,2,sp_record_payment,50002,Order is cancelled,"PM11,1004,100.00,UPI"
2,3,sp_record_payment,50003,Amount must be greater than zero,"PM12,1003,0.00,UPI"
3,4,sp_record_payment,50004,Payment exceeds order value,"PM13,1003,600.00,UPI"


In [8]:
run("DROP TABLE IF EXISTS emp_salary")
run("CREATE TABLE emp_salary (emp_id INT PRIMARY KEY, emp_name VARCHAR(50) NOT NULL, department VARCHAR(50) NOT NULL, salary INT NOT NULL)")
run("INSERT INTO emp_salary VALUES (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s), (%s,%s,%s,%s)", (1,"Aditi","Engineering",100000, 2,"Bharat","Engineering",100000, 3,"Chetan","Engineering",90000, 4,"Divya","Engineering",80000, 5,"Eshan","Engineering",80000, 6,"Farhan","Engineering",70000, 7,"Gauri","Sales",75000, 8,"Harsh","Sales",72000, 9,"Ira","Sales",72000, 10,"Jatin","Support",50000, 11,"Kavya","HR",60000, 12,"Lokesh","HR",60000))

run("""
SELECT emp_id,
       emp_name,
       department,
       salary,
       ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC, emp_id ASC) AS row_num,
       RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS rnk,
       DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dense_rnk
FROM emp_salary
ORDER BY department, salary DESC, emp_id
""")

run("""
WITH ranked_salaries AS (
    SELECT department,
           emp_name,
           salary,
           DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS salary_level
    FROM emp_salary
)
SELECT department, emp_name, salary, salary_level
FROM ranked_salaries
WHERE salary_level <= 3
ORDER BY department, salary DESC, emp_name
""")

,emp_id,emp_name,department,salary,row_num,rnk,dense_rnk
0,1,Aditi,Engineering,100000,1,1,1
1,2,Bharat,Engineering,100000,2,1,1
2,3,Chetan,Engineering,90000,3,3,2
3,4,Divya,Engineering,80000,4,4,3
4,5,Eshan,Engineering,80000,5,4,3
5,6,Farhan,Engineering,70000,6,6,4
6,11,Kavya,HR,60000,1,1,1
7,12,Lokesh,HR,60000,2,1,1
8,7,Gauri,Sales,75000,1,1,1
9,8,Harsh,Sales,72000,2,2,2


,department,emp_name,salary,salary_level
0,Engineering,Aditi,100000,1
1,Engineering,Bharat,100000,1
2,Engineering,Chetan,90000,2
3,Engineering,Divya,80000,3
4,Engineering,Eshan,80000,3
5,HR,Kavya,60000,1
6,HR,Lokesh,60000,1
7,Sales,Gauri,75000,1
8,Sales,Harsh,72000,2
9,Sales,Ira,72000,2


In [9]:
run("DROP VIEW IF EXISTS vw_book_author")
run("DROP TABLE IF EXISTS BookAuthors")
run("DROP TABLE IF EXISTS Authors")
run("DROP TABLE IF EXISTS Books")

run("CREATE TABLE Books (BookID INT PRIMARY KEY, Title VARCHAR(100) NOT NULL)")
run("CREATE TABLE Authors (AuthorID INT PRIMARY KEY, AuthorName VARCHAR(100) NOT NULL)")
run("CREATE TABLE BookAuthors (BookID INT NOT NULL, AuthorID INT NOT NULL, PRIMARY KEY (BookID, AuthorID), FOREIGN KEY (BookID) REFERENCES Books(BookID), FOREIGN KEY (AuthorID) REFERENCES Authors(AuthorID))")

run("INSERT INTO Books VALUES (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s)", (1,"Desert Winds", 2,"River Songs", 3,"The Glass Fort", 4,"Night Train", 5,"Salt and Stone", 6,"Paper Moons"))
run("INSERT INTO Authors VALUES (%s,%s), (%s,%s), (%s,%s), (%s,%s)", (1,"Meera Kaul", 2,"Rohan Sen", 3,"Tanvi Bose", 4,"Ishita Roy"))
run("INSERT INTO BookAuthors VALUES (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s)", (1,1, 2,2, 3,1, 3,3, 4,4, 5,2, 6,2, 6,1))

run("""
CREATE VIEW vw_book_author AS
SELECT b.BookID,
       b.Title AS Booktitle,
       GROUP_CONCAT(a.AuthorName ORDER BY a.AuthorName SEPARATOR ',') AS Authorname,
       GROUP_CONCAT(ac.book_count ORDER BY a.AuthorName SEPARATOR ',') AS Book_count
FROM Books AS b
JOIN BookAuthors AS ba ON ba.BookID = b.BookID
JOIN Authors AS a ON a.AuthorID = ba.AuthorID
JOIN (
    SELECT AuthorID, COUNT(DISTINCT BookID) AS book_count
    FROM BookAuthors
    GROUP BY AuthorID
) AS ac ON ac.AuthorID = a.AuthorID
GROUP BY b.BookID, b.Title
""")

run("SELECT Booktitle, Authorname, Book_count FROM vw_book_author ORDER BY BookID")

,Booktitle,Authorname,Book_count
0,Desert Winds,Meera Kaul,3
1,River Songs,Rohan Sen,3
2,The Glass Fort,"Meera Kaul,Tanvi Bose","3,1"
3,Night Train,Ishita Roy,1
4,Salt and Stone,Rohan Sen,3
5,Paper Moons,"Meera Kaul,Rohan Sen","3,3"


In [10]:
run("DROP VIEW IF EXISTS VW_EMP_DEPT")
run("DROP TABLE IF EXISTS Employee")
run("DROP TABLE IF EXISTS Dept_loc")

run("CREATE TABLE Employee (Empid INT PRIMARY KEY, Full_name VARCHAR(100) NOT NULL, Salary DECIMAL(12,2) NOT NULL, Hiredate DATE NOT NULL, deptid INT NULL)")
run("CREATE TABLE Dept_loc (deptid INT PRIMARY KEY, deptname VARCHAR(100) NOT NULL, loc VARCHAR(50) NOT NULL)")

run("INSERT INTO Employee VALUES (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s)", (1012,"Raji K Nair",35500,"2018-04-09",1, 1009,"Jeena",12000,"2018-11-17",2, 1005,"Nidhi R Shetty",10000,"2019-01-16",3, 1011,"Jiya Shetty",45500,"2019-03-27",4, 1003,"Arun Kumar Menon",45500,"2019-08-02",5, 1007,"Farah Ali",30000,"2020-02-10",None, 1001,"Manoj P Iyer",52000,"2020-07-15",2))
run("INSERT INTO Dept_loc VALUES (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s)", (1,"HR","Dallas", 2,"Sales","Chicago", 3,"Customer service","Chicago", 4,"IT","Boston", 5,"Research","Seattle", 6,"Legal","New York"))

run("""
CREATE VIEW VW_EMP_DEPT AS
SELECT x.Empid,
       CASE
           WHEN LOCATE(' ', TRIM(x.Full_name)) = 0 THEN TRIM(x.Full_name)
           ELSE CONCAT(SUBSTRING_INDEX(TRIM(x.Full_name), ' ', 1), ' ', SUBSTRING_INDEX(TRIM(x.Full_name), ' ', -1))
       END AS Full_name,
       x.Salary,
       CAST(ROUND(x.Salary * CASE
           WHEN d.loc = 'Chicago' THEN 1.05
           WHEN d.loc = 'Boston' THEN 1.03
           WHEN d.loc IN ('Dallas', 'New York') THEN 1.10
           ELSE 1.00
       END, 0) AS DECIMAL(12,0)) AS Inc_Sal,
       x.Previous_emp_sal,
       x.Hiredate,
       COALESCE(d.deptname, 'NA') AS deptname,
       COALESCE(d.loc, 'NA') AS loc,
       CASE
           WHEN x.Previous_emp_sal IS NULL THEN 'No previous employee'
           WHEN x.Salary > x.Previous_emp_sal THEN 'Salary is greater than previously hired employee'
           WHEN x.Salary < x.Previous_emp_sal THEN 'Salary is less than previously hired employee'
           ELSE 'Salary is same as previously hired employee'
       END AS message
FROM (
    SELECT Empid, Full_name, Salary, Hiredate, deptid,
           LAG(Salary) OVER (ORDER BY Hiredate, Empid) AS Previous_emp_sal
    FROM Employee
) AS x
LEFT JOIN Dept_loc AS d ON d.deptid = x.deptid
""")

run("SELECT * FROM VW_EMP_DEPT ORDER BY Hiredate, Empid")

,Empid,Full_name,Salary,Inc_Sal,Previous_emp_sal,Hiredate,deptname,loc,message
0,1012,Raji Nair,35500.00,39050,None,2018-04-09,HR,Dallas,No previous employee
1,1009,Jeena,12000.00,12600,35500.00,2018-11-17,Sales,Chicago,Salary is less than previously hired employee
2,1005,Nidhi Shetty,10000.00,10500,12000.00,2019-01-16,Customer service,Chicago,Salary is less than previously hired employee
3,1011,Jiya Shetty,45500.00,46865,10000.00,2019-03-27,IT,Boston,Salary is greater than previously hired employee
4,1003,Arun Menon,45500.00,45500,45500.00,2019-08-02,Research,Seattle,Salary is same as previously hired employee
5,1007,Farah Ali,30000.00,30000,45500.00,2020-02-10,NA,NA,Salary is less than previously hired employee
6,1001,Manoj Iyer,52000.00,54600,30000.00,2020-07-15,Sales,Chicago,Salary is greater than previously hired employee


In [11]:
run("DROP TABLE IF EXISTS monthly_sales")
run("CREATE TABLE monthly_sales (month_start DATE NOT NULL, region VARCHAR(20) NOT NULL, revenue DECIMAL(12,2) NOT NULL, PRIMARY KEY (month_start, region))")
run("INSERT INTO monthly_sales VALUES (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s), (%s,%s,%s)", ("2026-01-01","North",120000, "2026-02-01","North",135000, "2026-03-01","North",128000, "2026-04-01","North",150000, "2026-05-01","North",160000, "2026-06-01","North",145000, "2026-01-01","South",90000, "2026-02-01","South",85000, "2026-03-01","South",110000, "2026-04-01","South",105000, "2026-05-01","South",130000, "2026-06-01","South",140000))

run("""
SELECT region,
       month_start,
       revenue,
       SUM(revenue) OVER (
           PARTITION BY region
           ORDER BY month_start
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_total,
       ROUND(AVG(revenue) OVER (
           PARTITION BY region
           ORDER BY month_start
           ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
       ), 2) AS moving_avg_3m,
       ROUND(100 * revenue / SUM(revenue) OVER (PARTITION BY region), 2) AS pct_of_total
FROM monthly_sales
ORDER BY region, month_start
""")

,region,month_start,revenue,running_total,moving_avg_3m,pct_of_total
0,North,2026-01-01,120000.00,120000.00,120000.00,14.32
1,North,2026-02-01,135000.00,255000.00,127500.00,16.11
2,North,2026-03-01,128000.00,383000.00,127666.67,15.27
3,North,2026-04-01,150000.00,533000.00,137666.67,17.90
4,North,2026-05-01,160000.00,693000.00,146000.00,19.09
5,North,2026-06-01,145000.00,838000.00,151666.67,17.30
6,South,2026-01-01,90000.00,90000.00,90000.00,13.64
7,South,2026-02-01,85000.00,175000.00,87500.00,12.88
8,South,2026-03-01,110000.00,285000.00,95000.00,16.67
9,South,2026-04-01,105000.00,390000.00,100000.00,15.91


In [12]:
run("DROP TABLE IF EXISTS customer_updates")
run("CREATE TABLE customer_updates (load_id INT PRIMARY KEY, customer_id VARCHAR(5) NOT NULL, email VARCHAR(100) NOT NULL, city VARCHAR(50) NOT NULL, updated_at DATETIME NOT NULL)")
run("INSERT INTO customer_updates VALUES (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s), (%s,%s,%s,%s,%s)", (1,"C01","rahul@mail.com","Jaipur","2026-05-01 10:00:00", 2,"C02","priya@mail.com","Kochi","2026-05-01 10:05:00", 3,"C01","rahul.m@mail.com","Jaipur","2026-05-03 09:00:00", 4,"C03","arjun@mail.com","Kolkata","2026-05-02 12:00:00", 5,"C02","priya@mail.com","Kochi","2026-05-01 10:05:00", 6,"C01","rahul.m@mail.com","Delhi","2026-05-03 09:00:00", 7,"C04","fatima@mail.com","Mumbai","2026-05-04 08:30:00", 8,"C03","arjun.d@mail.com","Kolkata","2026-05-05 18:45:00"))

run("""
WITH ranked_updates AS (
    SELECT customer_id, email, city, updated_at, load_id,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY updated_at DESC, load_id DESC
           ) AS row_num
    FROM customer_updates
)
SELECT customer_id, email, city, updated_at, load_id
FROM ranked_updates
WHERE row_num = 1
ORDER BY customer_id
""")

run("""
WITH ranked_updates AS (
    SELECT customer_id,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY updated_at DESC, load_id DESC
           ) AS row_num
    FROM customer_updates
)
SELECT COUNT(*) AS rows_loaded,
       SUM(CASE WHEN row_num = 1 THEN 1 ELSE 0 END) AS rows_kept,
       SUM(CASE WHEN row_num > 1 THEN 1 ELSE 0 END) AS rows_removed
FROM ranked_updates
""")

,customer_id,email,city,updated_at,load_id
0,C01,rahul.m@mail.com,Delhi,2026-05-03 09:00:00,6
1,C02,priya@mail.com,Kochi,2026-05-01 10:05:00,5
2,C03,arjun.d@mail.com,Kolkata,2026-05-05 18:45:00,8
3,C04,fatima@mail.com,Mumbai,2026-05-04 08:30:00,7


,rows_loaded,rows_kept,rows_removed
0,8,4,4


In [13]:
run("DROP TABLE IF EXISTS logins")
run("CREATE TABLE logins (user_id VARCHAR(10) NOT NULL, login_date DATE NOT NULL, PRIMARY KEY (user_id, login_date))")
run("INSERT INTO logins VALUES (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s), (%s,%s)", ("U1","2026-03-01", "U1","2026-03-02", "U1","2026-03-03", "U1","2026-03-05", "U1","2026-03-06", "U1","2026-03-07", "U2","2026-03-01", "U2","2026-03-03", "U2","2026-03-04", "U2","2026-03-05", "U2","2026-03-06", "U2","2026-03-10", "U3","2026-03-02", "U4","2026-03-01", "U4","2026-03-02", "U4","2026-03-04", "U4","2026-03-05", "U4","2026-03-08", "U4","2026-03-09"))

run("""
WITH numbered_logins AS (
    SELECT user_id,
           login_date,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date) AS rn
    FROM logins
), login_groups AS (
    SELECT user_id,
           login_date,
           DATE_SUB(login_date, INTERVAL rn DAY) AS group_key
    FROM numbered_logins
), streaks AS (
    SELECT user_id,
           MIN(login_date) AS streak_start,
           MAX(login_date) AS streak_end,
           COUNT(*) AS streak_days
    FROM login_groups
    GROUP BY user_id, group_key
), ranked_streaks AS (
    SELECT user_id,
           streak_start,
           streak_end,
           streak_days,
           ROW_NUMBER() OVER (
               PARTITION BY user_id
               ORDER BY streak_days DESC, streak_end DESC
           ) AS streak_rank
    FROM streaks
)
SELECT user_id, streak_start, streak_end, streak_days
FROM ranked_streaks
WHERE streak_rank = 1
ORDER BY user_id
""")

,user_id,streak_start,streak_end,streak_days
0,U1,2026-03-05,2026-03-07,3
1,U2,2026-03-03,2026-03-06,4
2,U3,2026-03-02,2026-03-02,1
3,U4,2026-03-08,2026-03-09,2
